In [ ]:
#data
import numpy as np
from plotly.io import show
from sklearn.model_selection import train_test_split

from skfolio import Population, RiskMeasure
from skfolio.datasets import load_sp500_dataset
from skfolio.optimization import InverseVolatility, MeanRisk, ObjectiveFunction
from skfolio.preprocessing import prices_to_returns

prices = load_sp500_dataset()

X = prices_to_returns(prices)
X_train, X_test = train_test_split(X, test_size=0.33, shuffle=False)

print(X_train.head())

                AAPL       AMD       BAC       BBY       CVX        GE  \
Date                                                                     
1990-01-03  0.007576 -0.030303  0.008045  0.118056 -0.016229 -0.001876   
1990-01-04  0.003759 -0.015500 -0.021355 -0.012422 -0.012831 -0.005639   
1990-01-05  0.003745 -0.031996 -0.021821  0.000000 -0.014855 -0.009452   
1990-01-08  0.003731  0.000000  0.005633 -0.075472  0.009424  0.005725   
1990-01-09 -0.007435  0.016527  0.000000  0.000000 -0.007469 -0.020803   

                  HD       JNJ       JPM        KO       LLY       MRK  \
Date                                                                     
1990-01-03  0.003581  0.004072  0.033589 -0.014318  0.000000  0.015896   
1990-01-04  0.006244  0.002028  0.003991 -0.004993 -0.005557 -0.015647   
1990-01-05 -0.013298 -0.010408  0.003975 -0.008212 -0.010874 -0.020641   
1990-01-08 -0.009883  0.016944  0.000000  0.021159  0.000000  0.012839   
1990-01-09 -0.026316 -0.031026 -0.031

In [3]:
#model
model = MeanRisk(
    risk_measure=RiskMeasure.STANDARD_DEVIATION,
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    portfolio_params=dict(name="Max Sharpe"),
)
model.fit(X_train)
model.weights_

array([9.43837536e-02, 1.23703227e-07, 4.32481922e-08, 1.20892854e-01,
       3.18418329e-02, 7.69682669e-08, 1.78420643e-04, 1.24117994e-01,
       8.50336304e-08, 2.77970034e-02, 1.31617985e-07, 1.49536748e-07,
       1.16362392e-01, 5.73881398e-02, 9.91607330e-07, 1.09506312e-01,
       8.64772579e-02, 1.84018669e-01, 1.34639296e-02, 3.35698407e-02])

In [4]:
benchmark = InverseVolatility(portfolio_params=dict(name="Inverse Vol"))
benchmark.fit(X_train)
benchmark.weights_

array([0.03306735, 0.02548697, 0.03551377, 0.0296872 , 0.06358463,
       0.05434705, 0.04742354, 0.07049715, 0.03882539, 0.06697905,
       0.05570808, 0.05576851, 0.04723274, 0.06351213, 0.05581397,
       0.0676481 , 0.02564642, 0.03970752, 0.05744543, 0.06610498])

In [5]:
#prediction
pred_model = model.predict(X_test)
pred_bench = benchmark.predict(X_test)



In [6]:
np.asarray(pred_model)

array([ 0.00805138,  0.01084096,  0.00199137, ...,  0.00932288,
        0.00152751, -0.01787269])

In [7]:
print(pred_model.annualized_sharpe_ratio)
print(pred_bench.annualized_sharpe_ratio)

1.039972499946977
1.0036976120249754


In [8]:
#analysis
population = Population([pred_model, pred_bench])

In [9]:
population.plot_composition()

In [10]:
fig = population.plot_cumulative_returns()
show(fig)

In [11]:
population.summary()

,Max Sharpe,Inverse Vol
Mean,0.073%,0.064%
Annualized Mean,18.43%,16.06%
Variance,0.012%,0.010%
Annualized Variance,3.14%,2.56%
Semi-Variance,0.0063%,0.0053%
Annualized Semi-Variance,1.58%,1.33%
Standard Deviation,1.12%,1.01%
Annualized Standard Deviation,17.72%,16.00%
Semi-Deviation,0.79%,0.73%
Annualized Semi-Deviation,12.58%,11.54%


From the analysis on the test set, we see that the Maximum Sharpe Ratio portfolio outperform the inverse-volatility benchmark for the mean and the ratio measures including the Sharpe Ratio, and underperforms for the deviation and shortfall measures.